# 📚 06: Personalized Taste-Based Recommendations — Goal 3

## 🎯 Objective: The Personal Librarian — Synthesizing Taste & Content

This notebook implements **Goal 3: Hybrid Intelligence Engine** (Model-Based Collaborative Filtering). The system combines **user behavior (collaborative filtering)** with **content understanding (semantic embeddings)** to generate deeply personalized recommendations.

> *"This is the 'Master Brain' of the project — identifying the specific multidimensional intersection of a user's taste.

---

## 🧠 How It Works

| Step | Action | Description |
|------|--------|-------------|
| **1** | Load Components | User ratings (Goal 2) + Book embeddings (Goal 1) + Metadata |
| **2** | Create Embedding Lookup | Map book titles → 384-dim semantic vectors |
| **3** | Build User Profile Vector | Average embeddings of user's favorite books (≥4 stars) |
| **4** | Find Similar Users (KNN) | Locate 100 nearest neighbors based on rating patterns |
| **5** | Generate CF Candidates | Collect books neighbors liked (centered rating > 0) |
| **6** | Calculate Semantic Score | Cosine similarity between candidate book and user profile vector |
| **7** | Hybrid Scoring | `Final Score = (CF × 0.7) + (Semantic × 0.3)` |
| **8** | Aggressive Deduplication | Normalize titles (remove parentheses, punctuation, conjunctions) |
| **9** | Return Top N | Unique recommendations with hybrid confidence scores |

---

## 📊 Data Components

| Component | Source | Shape | Purpose |
|-----------|--------|-------|---------|
| **User Ratings** | `master_ratings_metadata.parquet` | 385,727 rows × 7 cols | User-book interactions (Goal 2) |
| **Book Metadata** | `books_metadata_only.parquet` | 209,452 rows × 9 cols | Book titles, authors, genres |
| **Embeddings** | `books_embeddings_matrix.npy` | 209,452 × 384 | Semantic vectors for each book (Goal 1) |

### Alignment Check

```text
✅ Metadata and embeddings are perfectly aligned (209,452 books each)
✅ Ratings filtered to 28,750 users × 18,335 books
✅ All rated books have corresponding embeddings

In [ ]:
# ============================================================
# CELL 1 - Loading the Hybrid Data
# ============================================================
import pandas as pd
import numpy as np

# 1. Load the User-Book interactions (Goal 2)
# Created in the EDA_Books_ratings.ipynb
df = pd.read_parquet("../Data/Books_rating/master_ratings_metadata.parquet")

# 2. Load the Labels/Metadata (Goal 1)
# This file MUST have the exact same number of rows as the .npy matrix
# and be in the exact same order as when the .npy was saved 
meta_df = pd.read_parquet("../Data/Books_data/books_metadata_only.parquet")

# 3. Load the raw Vectors (Goal 1)
embeddings_matrix = np.load("../Data/Books_data/books_embeddings_matrix.npy")

print(f"Ratings Loaded: {df.shape[0]:,} ratings")
print(f"Metadata Loaded: {meta_df.shape[0]:,} books")
print(f"Vectors Loaded:  {embeddings_matrix.shape[0]:,} vectors")

if meta_df.shape[0] == embeddings_matrix.shape[0]:
    print("Perfect Alignment! We can now map Titles to Vectors.")
else:
    print("Mismatch! We need to verify the row counts before proceeding.")

Ratings Loaded: 385,727 ratings
Metadata Loaded: 209,452 books
Vectors Loaded:  209,452 vectors
Perfect Alignment! We can now map Titles to Vectors.


In [2]:
print(df.shape)
df.head(10)

(385727, 7)


,User_id,Title,rating,review_date,review_year,authors_parsed,categories_parsed
0,A01038432MVI9JXYTTK5T,"20, 000 Leagues Under the Sea",5.0,2013-01-14,2013,[Jules Verne],[]
1,A01038432MVI9JXYTTK5T,"The annotated Jules Verne, Twenty thousand lea...",5.0,2013-01-14,2013,[],[]
2,A01038432MVI9JXYTTK5T,A JOURNEY TO THE CENTER OF THE EARTH,5.0,2013-01-14,2013,[Jules Verne],[Fiction]
3,A01038432MVI9JXYTTK5T,The Mysterious Island (Classics Illustrated),5.0,2013-01-14,2013,[Jules Verne],[]
4,A01038432MVI9JXYTTK5T,Twenty Thousand Leagues Under the Sea (Caxton ...,5.0,2013-01-14,2013,"[McClurg, Firm, Booksellers, Chicago]",[]
5,A0134066213WYQXLTVGYT,The Awakening,5.0,2012-12-17,2012,[Kate Chopin],[Fiction]
6,A0134066213WYQXLTVGYT,"The Awakening: Complete, Authoritative Text Wi...",5.0,2012-12-17,2012,[Kate Chopin],[Adultery]
7,A0134066213WYQXLTVGYT,Awakening: Kate Chopin Pb (Case Studies in Con...,5.0,2012-12-17,2012,[New York Public Library Staff],[]
8,A0134066213WYQXLTVGYT,Dracula (G. K. Hall (Large Print)),5.0,2012-12-17,2012,"[Bram Stoker, Grand Theatre Collection (Univer...",[High interest-low vocabulary books]
9,A0134066213WYQXLTVGYT,Public Secrets,5.0,2013-01-04,2013,[],[]


In [3]:
print(meta_df.shape)
meta_df.head(10)

(209452, 9)


,Title,description,authors_parsed,categories_parsed,year,image,infoLink,ratingsCount,content_for_embedding
0,The Alchemist,"""My heart is afraid that it will have to suffe...",[Paulo Coelho],[Fiction],2006.0,http://books.google.com/books/content?id=pTr44...,http://books.google.com/books?id=pTr44Sx6oWQC&...,4895.0,"the alchemist paulo coelho fiction ""my heart i..."
1,A Mission to Millions; The Story of Ernie Alle...,#1 NEW YORK TIMES BESTSELLER • NOW A MAJOR MOT...,[Laura Hillenbrand],[Biography & Autobiography],2014.0,http://books.google.com/books/content?id=1PeLD...,http://books.google.com/books?id=1PeLDQAAQBAJ&...,4572.0,a mission to millions; the story of ernie alle...
2,Unbroken,#1 NEW YORK TIMES BESTSELLER • NOW A MAJOR MOT...,[Laura Hillenbrand],[Biography & Autobiography],2014.0,http://books.google.com/books/content?id=1PeLD...,http://books.google.com/books?id=1PeLDQAAQBAJ&...,4572.0,unbroken laura hillenbrand biography & autobio...
3,Blue Like Jazz: Nonreligious Thoughts on Chris...,A popular minister recounts his zealous early ...,[Donald Miller],[Biography & Autobiography],2012.0,http://books.google.com/books/content?id=fNC3f...,http://books.google.nl/books?id=fNC3fdxYgZMC&d...,4562.0,blue like jazz: nonreligious thoughts on chris...
4,Eclipse,As the love triangle heats up in the third boo...,[Stephenie Meyer],[Young Adult Fiction],2007.0,http://books.google.com/books/content?id=lw99O...,https://play.google.com/store/books/details?id...,4392.0,eclipse stephenie meyer young adult fiction as...
5,Redeeming Love,NATIONAL BESTSELLER • NOW A MAJOR MOTION PICTU...,[Francine Rivers],[Fiction],2009.0,http://books.google.com/books/content?id=I7eMl...,https://play.google.com/store/books/details?id...,4249.0,redeeming love francine rivers fiction nationa...
6,A Walk in the Woods,Traces the author's adventurous trek along the...,[Bill Bryson],[Travel],1999.0,http://books.google.com/books/content?id=a-8nn...,http://books.google.nl/books?id=a-8nnowMab0C&d...,4234.0,a walk in the woods bill bryson travel traces ...
7,The Tipping Point: How Little Things Can Make ...,An introduction to the Tipping Point theory ex...,[Malcolm Gladwell],[Reference],2003.0,http://books.google.com/books/content?id=w6V_P...,http://books.google.com/books?id=w6V_PwAACAAJ&...,4218.0,the tipping point: how little things can make ...
8,"Travel Light, Eat Heavy Along the Appalachian ...","God only knows what possessed Bill Bryson, a r...",[Bill Bryson],[Travel],2012.0,http://books.google.com/books/content?id=WnyDg...,http://books.google.com/books?id=WnyDgV_vNhsC&...,4161.0,"travel light, eat heavy along the appalachian ..."
9,Henrietta,#1 NEW YORK TIMES BESTSELLER • “The story of m...,[Rebecca Skloot],[Science],2010.0,http://books.google.com/books/content?id=GFevO...,https://play.google.com/store/books/details?id...,4067.0,henrietta rebecca skloot science #1 new york t...


In [4]:
print(embeddings_matrix.shape)
# Let's look at the first vector to confirm it's loaded correctly
embeddings_matrix[0]

(209452, 384)


array([-1.32764608e-03,  9.65398364e-03,  4.03989255e-02,  8.99528489e-02,
       -5.07095791e-02, -2.07398441e-02, -3.82493883e-02, -1.87848303e-02,
        2.38286536e-02, -5.02375886e-02, -4.04486898e-03, -8.00675526e-02,
       -2.06992850e-02,  2.14526616e-02, -8.54798332e-02,  3.01074497e-02,
        1.50437998e-02, -4.68141250e-02,  5.89134078e-03,  1.32079404e-02,
        1.50364721e-02,  6.67499704e-03, -2.57642586e-02,  6.31241053e-02,
       -2.90627014e-02,  1.28074531e-02,  5.40055288e-03, -4.46268469e-02,
       -6.01670481e-02, -1.46198329e-02, -1.27894497e-02,  8.81545153e-03,
        2.56056357e-02, -2.96181310e-02, -7.76305869e-02, -7.65360193e-03,
        5.17134778e-02,  4.95916568e-02,  2.60413475e-02,  1.82091463e-02,
       -2.37192940e-02,  3.37933563e-02,  4.93680651e-04,  9.01771933e-02,
        1.50542939e-02, -1.00213565e-01,  1.71933603e-02, -2.14299690e-02,
        2.92827934e-02,  8.95163342e-02, -6.81652501e-02, -2.46127825e-02,
       -1.84830427e-02,  

In [5]:
# ============================================================
# CELL 2 - The Embedding Lookup Logic
# ============================================================

# Creates a dictionary (lookup table) that maps every book title to its row number in the embeddings matrix.
# {'The Alchemist': 0, 'Siddhartha': 1, '1984': 2}
title_to_row = {title: i for i, title in enumerate(meta_df['Title'])}

# Takes a book title and returns its 384-dimensional embedding vector
def get_book_vector(title):
    # Looks up the title in the dictionary. .get() returns None if title not found (instead of crashing)
    row_idx = title_to_row.get(title)
    if row_idx is not None:
        # Returns the vector at that specific row
        return embeddings_matrix[row_idx]
    return None

# Test check
test_title = meta_df['Title'].iloc[0]
print(f"Test lookup for '{test_title}':")
print(f"Vector shape: {get_book_vector(test_title).shape}")
print(f"{get_book_vector(test_title)[:5]} ...") # Print first 5 values of the vector for brevity

Test lookup for 'The Alchemist':
Vector shape: (384,)
[-0.00132765  0.00965398  0.04039893  0.08995285 -0.05070958] ...


### CELL 2: The Embedding Lookup Logic

### What It Does

This cell creates a **fast lookup system** to retrieve the 384-dimensional semantic vector (embedding) for any book title.

### How It Works

| Step | Action | Result |
|------|--------|--------|
| **1** | Build a dictionary mapping `{book_title: row_index}` using `enumerate()` | `{'The Alchemist': 0, 'Siddhartha': 1, ...}` |
| **2** | Define `get_book_vector(title)` function | Looks up title in dictionary and returns the embedding from `embeddings_matrix[row]` |
| **3** | Test with first book from metadata | Confirms function returns a vector of shape `(384,)` |

### Visual Flow

```text
Book Title (e.g., "The Alchemist")
         ↓
   title_to_row dictionary
         ↓
   "The Alchemist" → row 0
         ↓
   embeddings_matrix[0]
         ↓
   [0.12, -0.34, 0.56, ..., 0.78]  ← 384 numbers

In [6]:
# ============================================================
# CELL 3 - RE-CALCULATING THE CENTER 
# ============================================================

# 1. Calculate the average rating for each user
# Example: If User A rated 3 books with ratings [5, 4, 2], their mean would be (5+4+2)/3 = 3.67
user_means = df.groupby('User_id')['rating'].transform('mean')

# 2. Subtract the mean from the raw rating
# This creates the "Green Star / Red Star" logic again
df['rating_centered'] = df['rating'] - user_means

print("'rating_centered' column restored!")
df[['User_id', 'Title', 'rating', 'rating_centered']].head()

'rating_centered' column restored!


,User_id,Title,rating,rating_centered
0,A01038432MVI9JXYTTK5T,"20, 000 Leagues Under the Sea",5.0,0.0
1,A01038432MVI9JXYTTK5T,"The annotated Jules Verne, Twenty thousand lea...",5.0,0.0
2,A01038432MVI9JXYTTK5T,A JOURNEY TO THE CENTER OF THE EARTH,5.0,0.0
3,A01038432MVI9JXYTTK5T,The Mysterious Island (Classics Illustrated),5.0,0.0
4,A01038432MVI9JXYTTK5T,Twenty Thousand Leagues Under the Sea (Caxton ...,5.0,0.0


In [7]:
# ============================================================
# CELL 4 - Rebuilding the Sparse Matrix & KNN
# ============================================================
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# 1. Create the mappings
# Gets all unique user IDs from ratings
user_ids = df['User_id'].unique()
# Maps each user ID to a matrix row number
user_to_idx = {id: i for i, id in enumerate(user_ids)}

# Gets all unique book titles from ratings
book_titles_in_ratings = df['Title'].unique()
# Maps each book title to a matrix column number {'20,000 Leagues...': 0, 'The Hobbit': 1, ...}
book_to_idx_ratings = {title: i for i, title in enumerate(book_titles_in_ratings)}

# 2. Build the Sparse Matrix
# Row: User, Column: Book, Value: Centered Rating
user_item_matrix = csr_matrix((
    df['rating_centered'], 
    (df['User_id'].map(user_to_idx), df['Title'].map(book_to_idx_ratings))
))

# 3. Re-train the KNN (Super fast since the data is pre-cleaned)
# KNN.fit() does NOT modify user_item_matrix — it only stores a reference to the original matrix for fast read-only queries.
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=30, n_jobs=-1)
model_knn.fit(user_item_matrix) # Stores the matrix in memory

print(f"KNN Model Trained Successfully!")
print(f"Matrix Shape: {user_item_matrix.shape} (Users x Books)")

KNN Model Trained Successfully!
Matrix Shape: (28750, 18335) (Users x Books)


### CELL 4: Rebuilding the Sparse Matrix & KNN

### What It Does

This cell rebuilds the **collaborative filtering engine** by creating a sparse User-Item matrix and training the K-Nearest Neighbors (KNN) model for finding similar users.

### How It Works

| Step | Action | Result |
|------|--------|--------|
| **1** | Create mappings: convert User IDs and Book Titles to matrix indices (row/column numbers) | `user_to_idx` and `book_to_idx` dictionaries |
| **2** | Build sparse matrix using `csr_matrix` with `rating_centered` as values | Matrix shape: `(28,750 users × 18,335 books)` |
| **3** | Train KNN model with cosine similarity | Ready to find 30 most similar users |

### Key Parameters

| Parameter | Value | Why |
|-----------|-------|-----|
| `metric='cosine'` | Cosine similarity | Ignores unrated books (zeros), focuses on taste patterns |
| `algorithm='brute'` | Brute force | Best for sparse matrices |
| `n_neighbors=30` | 30 neighbors | Balance between signal quality and noise |
| `n_jobs=-1` | All CPU cores | Faster computation |

In [8]:
# ============================================================
# CELL 5 - The Hybrid Recommendation Function with Aggressive Deduplication
# ============================================================

import re # Regular expressions for cleaning book titles
import numpy as np

def get_hybrid_recommendations(target_user_id, n_recs=5, cf_weight=0.7, sem_weight=0.3):
    """
    Hybrid Recommender: Combines Collaborative Filtering (KNN) with Semantic Meanings (Embeddings).
    
    Note: Aggressive deduplication is included as a safety net, 
    even though EDA already cleaned most duplicates. This prevents
    edge cases where different title strings for the same book
    appear in recommendation candidates.
    """
    if target_user_id not in user_to_idx:
        print(f"❌ User {target_user_id} not found. User should have at least 5 ratings to receive recommendations.")
        return []

    # 1. GET USER HISTORY
    user_data = df[df['User_id'] == target_user_id]
    already_read = set(user_data['Title']) 
    
    # 2. CREATE THE "USER SOUL VECTOR" (Profile Vector)
    # Target high-rated books first for a cleaner taste profile
    favorite_books = user_data[user_data['rating'] >= 4]['Title'].tolist()
    fav_vectors = [get_book_vector(t) for t in favorite_books if get_book_vector(t) is not None] # Converts each favorite book - its 384-dim embedding vector
    
    if not fav_vectors: 
        fav_vectors = [get_book_vector(t) for t in user_data['Title'].tolist() if get_book_vector(t) is not None]
        
    user_profile_vector = np.mean(fav_vectors, axis=0).reshape(1, -1) # Averages all vectors - single 384-dim "taste profile"

    # 3. GENERATE CANDIDATES (Collaborative Filtering)
    user_idx = user_to_idx[target_user_id]
    # 100 neighbors to ensure we have plenty of unique options after filtering clones
    distances, indices = model_knn.kneighbors(user_item_matrix[user_idx], n_neighbors=100)
    
    neighbor_indices = indices.flatten()[1:] # Extracts neighbor indices, skips the user themselves
    neighbor_distances = distances.flatten()[1:]
    
    candidate_scores = {} 

    for i in range(len(neighbor_indices)):
        sim_to_neighbor = 1 - neighbor_distances[i]
        n_id = user_ids[neighbor_indices[i]]
        n_ratings = df[df['User_id'] == n_id]
        liked_by_neighbor = n_ratings[n_ratings['rating_centered'] > 0] # Filters for books neighbor liked ABOVE their average
        
        for _, row in liked_by_neighbor.iterrows():
            book = row['Title']
            if book not in already_read:
                if book not in candidate_scores:
                    book_vec = get_book_vector(book) # Retrieves embedding for the candidate book
                    sem_sim = 0
                    if book_vec is not None:
                        # How similar is this book to user's taste?
                        sem_sim = cosine_similarity(user_profile_vector, book_vec.reshape(1, -1))[0][0]
                    
                    candidate_scores[book] = {'cf': 0, 'sem': sem_sim}
                
                candidate_scores[book]['cf'] += row['rating_centered'] * sim_to_neighbor # Weighted by similarity to neighbor for a book (different neighbors can contribute to the same book's CF score)

    # 4. HYBRID RANKING & DEDUPLICATION
    final_ranking = []
    seen_base_titles = set()

    # Sort all candidates by the weighted hybrid (for each book, calculate: CF×0.7 + SEM×0.3, then sort by this score)
    sorted_candidates = sorted(
        candidate_scores.items(), 
        key=lambda x: (x[1]['cf'] * cf_weight) + (x[1]['sem'] * sem_weight),  # Sorts by hybrid score (CF×0.7 + SEM×0.3)
        reverse=True
    ) # [('The Fountainhead', {'cf': 3.8, 'sem': 0.92}),...] # Hybrid: ~2.93
    
    # Iterates through candidates from highest to lowest score
    for book, scores in sorted_candidates:
        # STEP A: Remove everything in parentheses (e.g., "Book (Audiobook)" -> "Book")
        clean_name = re.sub(r'\(.*\)', '', book)
        
        # STEP B: Remove specific conjunctions that cause duplicate entries
        # We replace 'and' with empty string to match against versions using '&'
        fingerprint = clean_name.lower().replace('and', '')
        
        # STEP C: Strip all non-alphanumeric characters (spaces, symbols, etc.)
        fingerprint = "".join(filter(str.isalnum, fingerprint))
        
        # STEP D: Add to list only if this "fingerprint" hasn't been seen yet
        if fingerprint not in seen_base_titles:
            h_score = (scores['cf'] * cf_weight) + (scores['sem'] * sem_weight) # Calculates final hybrid score
            final_ranking.append((book, h_score)) # Adds tuple (title, score) to results
            seen_base_titles.add(fingerprint) # Marks this fingerprint as seen
        
        if len(final_ranking) >= n_recs:
            break

    return final_ranking # [(Title_1, Score_1), (Title_2, Score_2), ...] List of tuples

print("Pipeline Complete: Integrated Collaborative Filtering + NLP Transformers | Variant Filter: ON")

Pipeline Complete: Integrated Collaborative Filtering + NLP Transformers | Variant Filter: ON


### CELL 5: The Hybrid Recommendation Function

### What It Does

This function combines **Collaborative Filtering (crowd wisdom)** with **Semantic Understanding (content matching)** to generate personalized book recommendations. It also applies aggressive title deduplication to prevent duplicate recommendations.

### The Hybrid Formula

```text
Final Score = (CF Score × 0.7) + (Semantic Score × 0.3)

- CF Score = What similar users liked (weighted by similarity)
- Semantic Score = How well the book matches user's taste profile
```
### Steps

| Step | Action | Details |
|------|--------|---------|
| **1** | Get user history | Extract books the user already read (to filter later) |
| **2** | Create "User Soul Vector" | Average embeddings of user's favorite books (≥4 stars) → 384-dim taste profile |
| **3** | Find similar users (KNN) | Locate 30 nearest neighbors based on rating patterns |
| **4** | Collect candidate books | Gather books neighbors liked (centered rating > 0) that user hasn't read |
| **5** | Calculate semantic score | For each candidate, compute cosine similarity with user profile vector |
| **6** | Hybrid ranking | Combine CF and Semantic scores with weights (70/30) |
| **7** | Aggressive deduplication | Normalize titles (remove parentheses, punctuation, conjunctions) to merge duplicates |
| **8** | Return top N | Return unique recommendations with hybrid scores [(Title_1, Score_1), (Title_2, Score_2), ...] |

In [9]:
# ============================================================
# CELL 6 - Testing the Hybrid System (With User Profile)
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

def test_hybrid_system(user_id):
    if user_id not in user_to_idx:
        return print(f"❌ User {user_id} not found. User should have at least 5 ratings to receive recommendations.")

    print(f"ANALYZING HYBRID RECOMMENDATIONS FOR: {user_id}")
    
    # --- PART 1: USER PROFILE ---
    # Pull the history from the main dataframe
    user_history = df[df['User_id'] == user_id].sort_values(by='rating', ascending=False)
    
    print(f"📖 USER PROFILE")
    print(f"Total unique books rated: {len(user_history)}")
    print("-" * 80)
    print(f"{'TITLE':<55} | {'RATING':<7} | {'GENRE'}")
    
    for _, row in user_history.head(10).iterrows():
        genre = row['categories_parsed'][0] if len(row['categories_parsed']) > 0 else "Unknown"
        print(f"{str(row['Title'])[:53]:<55} | {row['rating']:<7} | {genre}")
    
    print("\n" + "-"*80 + "\n")

    # --- PART 2: HYBRID RECOMMENDATIONS ---
    # We use our hybrid engine (70% Behavior, 30% Meaning)
    recs = get_hybrid_recommendations(user_id, cf_weight=0.7, sem_weight=0.3, n_recs=5) # Returns a list of tuples: [(Title_1, Score_1), (Title_2, Score_2), ...]
    
    if not recs:
        return print("No recommendations found for this user.")

    print(f"TOP 5 HYBRID RECOMMENDATIONS (Behavior + Meaning):")
    print("-" * 80)
    print(f"{'#':<3} | {'BOOK TITLE':<55} | {'SCORE'}")
    print("-" * 80)
    
    for i, (book, score) in enumerate(recs, 1):
        # Fetch metadata from meta_df
        meta = meta_df[meta_df['Title'] == book].iloc[0] if book in meta_df['Title'].values else None # This is a safety check in case the recommended book isn't found in the metadata (should be rare due to our data alignment)
        
        # Safety check for author and genre
        if meta is not None:
            author = meta['authors_parsed'][0] if len(meta['authors_parsed']) > 0 else "Unknown Author"
            genre = meta['categories_parsed'][0] if len(meta['categories_parsed']) > 0 else "Unknown"
        else:
            author, genre = "Unknown Author", "Unknown"
            
        print(f"{i:<3} | {book[:53]:<55} | {score:.4f}")
        print(f"    └─ Author: {author} | Genre: {genre}")
    print("-"*80)

### CELL 6: Testing the Hybrid System

### What It Does

This cell tests the complete hybrid recommendation engine by displaying a user's reading profile and generating personalized recommendations using both collaborative filtering (70%) and semantic understanding (30%).

### How It Works (Step by Step)

| Step | Action | Details |
|------|--------|---------|
| **1** | Validate user | Checks if user exists in the matrix (has at least 5 ratings) |
| **2** | Display user profile | Shows user's top 10 highest-rated books with titles, ratings, and genres |
| **3** | Generate hybrid recommendations | Calls `get_hybrid_recommendations()` with 70% CF / 30% Semantic weights |
| **4** | Fetch metadata | Retrieves author and genre information for each recommended book |
| **5** | Display results | Shows top 5 recommendations with rank, title, score, author, and genre |

### Key Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `cf_weight` | 0.7 | Weight for collaborative filtering (crowd wisdom) |
| `sem_weight` | 0.3 | Weight for semantic similarity (content matching) |
| `n_recs` | 5 | Number of recommendations to return |


In [10]:
# ============================================================
# CELL 7 - FINDING A TARGET USER WITH 10+ RATINGS (For Testing The Function)
# ============================================================

# 1. Count ratings per user
user_activity = df['User_id'].value_counts()

# 2. Filter for users with exactly 10 ratings
active_users = user_activity[user_activity == 10]

print(f"Total users with 10 ratings: {len(active_users):,}")
print("\nHere are 10 User_ids you can test:")
print(active_users.head(10))

# 3. Pick a random one from this list to test your function
sample_user = active_users.index[5] # Picking the 6th one in the list
print(f"\nTry testing with this User_id: {sample_user}")

Total users with 10 ratings: 1,165

Here are 10 User_ids you can test:
User_id
A10250WFWWKGWV    10
A102P9UKBY9P75    10
A102VPNZTRP1YA    10
A103YPVK70AGXR    10
A10JQQVC4L91RD    10
A10R81SKUK3TG6    10
A10SAVQUXJ7OY1    10
A10TR7FH7DXFA2    10
A10WE4JQOHF6H2    10
A10Z51XLV63OA6    10
Name: count, dtype: int64

Try testing with this User_id: A10R81SKUK3TG6


In [11]:
# ============================================================
# CELL 8 - Testing the Hybrid System (With User Profile)
# ============================================================
test_hybrid_system("A10R81SKUK3TG6")

ANALYZING HYBRID RECOMMENDATIONS FOR: A10R81SKUK3TG6
📖 USER PROFILE
Total unique books rated: 10
--------------------------------------------------------------------------------
TITLE                                                   | RATING  | GENRE
The Alchemist: A Fable about Following Your Dream       | 5.0     | Unknown
Aunt Julia And The Scriptwriter                         | 5.0     | Fiction
It's Not About the Bike: My Journey Back to Life        | 5.0     | Biography & Autobiography
Jonathan Livingston Seagull                             | 5.0     | Fables
Jonathan Livingston Seagull: A Story                    | 5.0     | Fables
Who Moved My Cheese? An-Amazing Way to Deal with Chan   | 5.0     | Business & Economics
Wine For Dummies (For Dummies (Lifestyles Paperback))   | 5.0     | Cooking
The Persian Puzzle: The Conflict Between Iran and Ame   | 4.0     | Political Science
The Pilgrimage: A Contemporary Quest for Ancient Wisd   | 4.0     | Christian pilgrims and pilgrimage